# Faruq-v3 DC2 predicted raw RGB local-stream screen — v2

Development-validation broad-search screen for the predicted-box raw RGB local stream inspired by DC2. The detector remains frozen, matched objects are cropped from the original RGB image, and a dedicated MobileNetV3-Small local classifier is compared against the detector's native class decision on the same matched validation objects. Crop resolution is frozen at **128 x 128 independently of the coffee-validation-selected DC2a best resolution**. The locked holdout remains unavailable and unopened. This is not detector mAP and not yet full DC2/MSFA.

Protocol: `faruq-v3-dc2-predicted-raw-crop-screening-v2`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/dc2-predicted-raw-crop-screening'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('REPO:', REPO)
print('BRANCH:', BRANCH)

In [ ]:
import json, torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-acmc-one-stage-v1/ACMC1_seed42/weights/best.pt',
    'experiments/faruq-v3-dc2-raw-crop-resolution-search-v1/dc2_raw_crop_resolution_screening.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
DETECTOR = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-one-stage-v1/ACMC1_seed42/weights/best.pt')
RAW_CROP_SUMMARY = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-dc2-raw-crop-resolution-search-v1/dc2_raw_crop_resolution_screening.json')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-dc2-predicted-raw-crop-screening-v2'
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'Locked holdout tidak boleh tersedia.'
dc2a = json.loads(RAW_CROP_SUMMARY.read_text(encoding='utf-8'))
assert dc2a['protocol'] == 'faruq-v3-dc2-raw-crop-resolution-search-v1'
assert dc2a['decision'] == 'RETAIN_DC2_LOCAL_STREAM'
assert dc2a['next_action'] == 'AUTHORIZE_PREDICTED_RAW_CROP_LOCAL_STREAM_INTEGRATION'
assert dc2a['test_images_accessed'] is False
print('GPU       :', torch.cuda.get_device_name(0))
print('PROJECT   :', PROJECT_ROOT)
print('DATA      :', DATA_ROOT)
print('DETECTOR  :', DETECTOR)
print('DC2a      :', RAW_CROP_SUMMARY)
print('DC2a BEST :', dc2a['best_resolution'], '(audit only; NOT reused)')
print('DC2b CROP : 128 (paper-frozen before this screen)')
print('OUTPUT    :', OUTPUT_ROOT)

In [ ]:
command = [
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_dc2_crop.py',
    'tests/test_dc2_predicted_crop.py',
]
print('STATIC CHECK:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)
print('PASS: raw crop semantics, class-agnostic matching, paired GT/predicted crop control, frozen 128 resolution, and gate verified.')

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_dc2_predicted_crop_screening',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED_SUMMARY),
    '--detector-checkpoint', str(DETECTOR),
    '--raw-crop-summary', str(RAW_CROP_SUMMARY),
    '--output-root', str(OUTPUT_ROOT),
    '--seed', '42', '--device', '0', '--epochs', '20', '--batch-size', '64', '--workers', '2',
    '--authorize-training',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.run(command, cwd=REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(process.stdout, end='', flush=True)
if process.returncode != 0:
    raise RuntimeError(f'DC2 predicted-crop screen gagal dengan return code {process.returncode}; traceback lengkap tercetak di atas.')

In [ ]:
import pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'dc2_predicted_raw_crop_screening.json'
assert SUMMARY.is_file(), f'DC2 predicted-crop screen belum selesai: {SUMMARY}'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['protocol'] == 'faruq-v3-dc2-predicted-raw-crop-screening-v2'
assert result['evaluation_split'] == 'development_val_detector_matched_targets'
assert result['test_images_accessed'] is False
assert result['resolution'] == 128
assert result['resolution_source'] == 'paper_frozen_dc2_followup_128_not_coffee_val_selected'
native = result['results']['native_detector_on_matched_val']
gt = result['results']['gt_crop_local_on_same_matched_objects']['metrics']
pred = result['results']['predicted_crop_local']['metrics']
rows = [
    {'arm': 'native_detector_matched', **{k: native[k] for k in ('accuracy','macro_f1','bottom3_f1','worst_f1')}},
    {'arm': 'gt_raw_crop_matched', **{k: gt[k] for k in ('accuracy','macro_f1','bottom3_f1','worst_f1')}},
    {'arm': 'predicted_raw_crop', **{k: pred[k] for k in ('accuracy','macro_f1','bottom3_f1','worst_f1')}},
]
frame = pd.DataFrame(rows)
display(frame.style.format({'accuracy': '{:.2%}', 'macro_f1': '{:.2%}', 'bottom3_f1': '{:.2%}', 'worst_f1': '{:.2%}'}))
print('RESOLUTION      :', result['resolution'], result['resolution_source'])
print('DC2a BEST UNUSED:', result['dc2a_best_resolution_observed_but_not_reused'])
print('COVERAGE        :', result['coverage'])
print('DELTA vs native :', result['deltas_vs_native_matched'])
print('RETENTION vs GT :', result['retention_vs_gt_matched'])
print('CRITERIA        :', result['criteria'])
print('DECISION        :', result['decision'])
print('NEXT            :', result['next_action'])
print('SUMMARY         :', SUMMARY)
print('Catatan: ini klasifikasi objek yang berhasil dipasangkan pada development-val, bukan detector mAP, bukan independent confirmation, dan belum full DC2/MSFA.')